# Bronze — Statistics Iceland Ingestion
Fetches GDP (annual change by quarter), the market price index of residential housing, the wage index and the unemployment rate from Statistics Iceland via the PX-Web REST API.

**Source:** Statistics Iceland PX-Web API  
**Datasets:** THJ01601 (GDP), VIS01106 (market price of residential housing), LAU04000 (Wage Index), VIN00003 (Labour Force Survey, monthly trend)  
**Output:** `bronze.statistics_iceland_raw` (Delta table) — one row per observation, `dataset` column identifies the series  
**Load mode:** `load_mode` = `incremental` (default) requests only the last 24 months / 8 quarters with a PX-Web `top` filter and appends; `full` requests the whole history and overwrites. Silver keeps the latest version of each observation.


In [ ]:
# Parameters (set by the pipeline; toggle "parameter cell" in Fabric if it shows as a normal cell)
# "incremental": load only recent periods and append; "full": reload the whole history
load_mode = "incremental"


In [ ]:
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Retry session — handles 429 rate limiting automatically
session = requests.Session()
retry = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504]
)
session.mount("https://", HTTPAdapter(max_retries=retry))
TIMEOUT_SECONDS = 120

MONTHS_BACK = 24    # incremental: re-read the last 24 months, so revisions are picked up
QUARTERS_BACK = 8   # and the last 8 quarters for GDP

if load_mode == "incremental" and spark.catalog.tableExists("bronze.statistics_iceland_raw"):
    is_full_load = False
    print(f"Incremental load: last {MONTHS_BACK} months and {QUARTERS_BACK} quarters")
else:
    is_full_load = True
    print("Full load: whole history")

def time_filter(url, periods):
    """Incremental only: a PX-Web 'top' filter on the table's time variable (found from its metadata)."""
    if is_full_load:
        return []
    meta = session.get(url, timeout=TIMEOUT_SECONDS).json()
    time_var = next(v for v in meta["variables"] if v.get("time"))
    return [{"code": time_var["code"], "selection": {"filter": "top", "values": [str(periods)]}}]


In [ ]:
# GDP — Quarterly YoY Growth (THJ01601)
GDP_URL = "https://px.hagstofa.is/pxis/api/v1/is/Efnahagur/thjodhagsreikningar/landsframl/2_landsframleidsla_arsfj/THJ01601.px"

resp_gdp = session.post(GDP_URL, json={
    "query": [
        {"code": "Mælikvarði", "selection": {"filter": "item", "values": ["2"]}},
        {"code": "Skipting",   "selection": {"filter": "item", "values": ["14"]}},
    ] + time_filter(GDP_URL, QUARTERS_BACK),
    "response": {"format": "json"},
}, timeout=TIMEOUT_SECONDS)

if resp_gdp.status_code == 400:
    print(f"GDP API Error Detail: {resp_gdp.text}")

resp_gdp.raise_for_status()

gdp_records = [
    {
        "dataset": "gdp", 
        "period_raw": i["key"][2], # Quarter is at index [2]
        "value_raw": float(i["values"][0]), 
        "ingested_at": pd.Timestamp.now(tz="UTC")
    }
    for i in resp_gdp.json()["data"]
    if i["values"][0] not in (".", "", None)
]

In [ ]:
# Market price index of residential housing, whole country (VIS01106)
HPI_URL = "https://px.hagstofa.is/pxis/api/v1/is/Efnahagur/visitolur/1_vnv/3_greiningarvisitolur/VIS01106.px"

resp_hpi = session.post(HPI_URL, json={
    "query": [
        {
            "code": "Vísitala", 
            "selection": {"filter": "item", "values": ["RI_total"]}
        }
    ] + time_filter(HPI_URL, MONTHS_BACK),
    "response": {"format": "json"}
}, timeout=TIMEOUT_SECONDS)

if resp_hpi.status_code == 400:
    print(f"HPI API Error Detail: {resp_hpi.text}")

resp_hpi.raise_for_status()

hpi_records = [
    {
        "dataset": "hpi", 
        "period_raw": i["key"][0], # Month is at index [0]
        "value_raw": float(i["values"][0]),
        "ingested_at": pd.Timestamp.now(tz="UTC")
    }
    for i in resp_hpi.json()["data"]
    if i["values"][0] not in (".", "", None)
]

In [ ]:
# Wage Index (LAU04000)
WAGE_URL = "https://px.hagstofa.is/pxis/api/v1/is/Samfelag/launogtekjur/2_lvt/1_manadartolur/LAU04000.px"

resp_wages = session.post(WAGE_URL, json={
    "query": [
        {"code": "Eining", "selection": {"filter": "item", "values": ["index"]}}
    ] + time_filter(WAGE_URL, MONTHS_BACK),
    "response": {"format": "json"}
}, timeout=TIMEOUT_SECONDS)

if resp_wages.status_code == 400:
    print(f"Wage API Error Detail: {resp_wages.text}")

resp_wages.raise_for_status()

wage_records = [
    {
        "dataset": "wage_index", 
        "period_raw": i["key"][0], # Month is at index [0]
        "value_raw": float(i["values"][0]), 
        "ingested_at": pd.Timestamp.now(tz="UTC")
    }
    for i in resp_wages.json()["data"]
    if i["values"][0] not in (".", "", None)
]

In [ ]:
# Unemployment rate — Labour Force Survey, monthly trend (VIN00003)
# Value codes are looked up by their published labels, so the query doesn't depend on internal codes.
UNEMP_URL = "https://px.hagstofa.is/pxis/api/v1/is/Samfelag/vinnumarkadur/vinnumarkadsrannsokn/1_manadartolur/VIN00003.px"

meta = session.get(UNEMP_URL, timeout=TIMEOUT_SECONDS)
meta.raise_for_status()
variables = meta.json()["variables"]

def value_code(variable_text, value_text):
    var = next(v for v in variables if v["text"] == variable_text)
    return var["code"], var["values"][var["valueTexts"].index(value_text)]

sex_code, sex_value = value_code("Kyn/aldur", "Alls")
unit_code, unit_value = value_code("Eining", "Atvinnuleysi")
month_index = next(i for i, v in enumerate(variables) if v.get("time"))

resp_unemp = session.post(UNEMP_URL, json={
    "query": [
        {"code": sex_code,  "selection": {"filter": "item", "values": [sex_value]}},
        {"code": unit_code, "selection": {"filter": "item", "values": [unit_value]}},
    ] + time_filter(UNEMP_URL, MONTHS_BACK),
    "response": {"format": "json"},
}, timeout=TIMEOUT_SECONDS)

if resp_unemp.status_code == 400:
    print(f"Unemployment API Error Detail: {resp_unemp.text}")

resp_unemp.raise_for_status()

unemployment_records = [
    {
        "dataset": "unemployment_rate",
        "period_raw": i["key"][month_index],
        "value_raw": float(i["values"][0]),
        "ingested_at": pd.Timestamp.now(tz="UTC")
    }
    for i in resp_unemp.json()["data"]
    if i["values"][0] not in (".", "", None)
]

In [ ]:
# Source check: print each table's official title and the labels of the values selected,
# so the series used are documented in Statistics Iceland's own words.
SELECTIONS = {
    GDP_URL:   {"Mælikvarði": "2", "Skipting": "14"},
    HPI_URL:   {"Vísitala": "RI_total"},
    WAGE_URL:  {"Eining": "index"},
    UNEMP_URL: {"Kyn/aldur": sex_value, "Eining": unit_value},
}
for url, selected in SELECTIONS.items():
    meta = session.get(url, timeout=TIMEOUT_SECONDS).json()
    print(meta["title"])
    for var in meta["variables"]:
        if var["code"] in selected:
            label = var["valueTexts"][var["values"].index(selected[var["code"]])]
            print(f"    {var['text']}: {label}")


In [ ]:
new_data = pd.DataFrame(gdp_records + hpi_records + wage_records + unemployment_records)
spark_df = spark.createDataFrame(new_data)

if is_full_load:
    spark_df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable("bronze.statistics_iceland_raw")
    print(f"Full load: {spark_df.count()} rows")
else:
    spark_df.write.format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable("bronze.statistics_iceland_raw")
    print(f"Appended {spark_df.count()} rows")